<div style="padding: 20px; background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🖴️ Module 4.1: Vector Store Foundations & Comparison</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Understanding where embeddings live and how to search them efficiently.</p>
</div>

---

## 1. What is a Vector Database?

Traditional databases (SQL, MongoDB) search for **exact matches** (e.g., `WHERE word = 'apple'`). 
A **Vector Database** searches for **semantic similarity** (e.g., `FIND VECTORS CLOSEST TO 'apple'`).

Vector databases are purpose-built to store the high-dimensional arrays (embeddings) we generated in Module 3 and perform lightning-fast **Approximate Nearest Neighbor (ANN)** searches across millions of records.

### The Landscape
| Store | Type | Best For | Local/Free? |
| :--- | :--- | :--- | :--- |
| **ChromaDB** | Embedded DB | Learning, local apps, prototyping | ✅ 100% Free/Local |
| **FAISS** | Library (Meta) | High-performance in-memory search | ✅ 100% Free/Local |
| **Pinecone** | Managed Cloud | Zero-ops production deployment | ❌ Paid (small free tier) |
| **Qdrant** | Server/Rust | High-scale production | ✅ Open Source |

> [!TIP]
> For this curriculum, we strictly use **Chroma** and **FAISS** because they run entirely on your local machine with zero setup and zero cost.

### Course alignment and free-first stack

- Covers: Vector-store concepts, local Chroma, FAISS, and indexing trade-offs.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
# Setup: Let's import our tools and initialize our FREE local embeddings.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import time
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Using the lightweight HuggingFace model we learned about in Module 3
print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))
print("Model loaded successfully!")

## 2. Ingesting Data

Let's create a small corpus of documents to store. Notice that we attach **metadata** to each document. Metadata is crucial for pre-filtering results before doing vector math.

In [ ]:
docs = [
    Document(page_content="Chroma is an open-source embedding database favored for local dev.", metadata={"tool": "chroma"}),
    Document(page_content="FAISS is a library developed by Meta for efficient similarity search.", metadata={"tool": "faiss"}),
    Document(page_content="Qdrant is written in Rust and supports advanced payload filtering.", metadata={"tool": "qdrant"}),
    Document(page_content="Pinecone is a cloud-native vector database service.", metadata={"tool": "pinecone"}),
    Document(page_content="Milvus is designed for billion-scale vector workloads.", metadata={"tool": "milvus"}),
]

query = "Which tool is built by Meta?"

## 3. FAISS vs Chroma implementation

Let's see how both of these popular local tools handle ingestion and search.

In [ ]:
# --- FAISS (Facebook AI Similarity Search) ---
# FAISS is technically an index, not a full database. It lives purely in memory.
t0 = time.time()
faiss_index = FAISS.from_documents(docs, embeddings)
faiss_time = time.time() - t0

faiss_results = faiss_index.similarity_search(query, k=1)

print(f"[FAISS] Ingested in {faiss_time:.4f}s")
print(f"[FAISS] Best Match for '{query}':")
print(f"        -> {faiss_results[0].page_content}\n")

# --- ChromaDB ---
# Chroma acts more like a traditional database. It can be saved to disk easily.
t0 = time.time()
chroma_db = Chroma.from_documents(docs, embeddings, collection_name="demo_collection")
chroma_time = time.time() - t0

chroma_results = chroma_db.similarity_search(query, k=1)

print(f"[Chroma] Ingested in {chroma_time:.4f}s")
print(f"[Chroma] Best Match for '{query}':")
print(f"         -> {chroma_results[0].page_content}")

## Summary
- **FAISS** is generally faster for raw ingestion and searching because it's highly optimized C++ code running purely in memory.
- **Chroma** offers a more friendly developer experience, robust metadata handling, and easy persistence (which we will explore in the next notebook).